# Hybrid Digital Twin for Li-ion Batteries — Python

**Can you predict when a battery reaches end of life, from only the first 40 % of its life?**

Three models, one honest test:

| | Model | What it knows |
|---|---|---|
| 1 | **Physics** | a degradation law from the literature, two parameters |
| 2 | **Data** | a Gaussian process. Flexible, and has no idea what a battery is |
| 3 | **Hybrid** | the physics, plus a small network fitted to what the physics gets wrong |

The test is a **temporal split**: fit on the first 40 % of cycles, forecast the rest. Not a random split — a random split trains on Tuesday and Thursday to predict Wednesday, which is interpolation, and is not the question anyone is asking.

> **Runs in Colab as-is.** Python is Colab's native runtime — press Run all.

---

In [ ]:
# Everything below runs on the standard Colab image. No installs needed.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.neural_network import MLPRegressor

sns.set_theme(style='whitegrid', context='talk', font_scale=0.85)
SEED, TRAIN_FRACTION, EOL = 0, 0.40, 0.80
np.random.seed(SEED)

## 0. The data

NASA Ames Prognostics Center, Li-ion battery aging dataset — four cells (B0005, B0006, B0007, B0018) cycled to failure at 24 °C. We ship the capacity series, not the 21 MB of raw measurements.

In [ ]:
# Local checkout first, GitHub second. In Colab there is no local copy, so it
# downloads; in CI the repository is already checked out, so the notebook is
# tested against the data that ships with it rather than against whatever
# happens to be on main. Both candidates are tried because a notebook's working
# directory is sometimes the repo root and sometimes notebooks/.
import os

URL = "https://raw.githubusercontent.com/otwin-core/otwin-hybrid/main/data/battery_soh.csv"
SOURCE = next((p for p in ("data/battery_soh.csv", "../data/battery_soh.csv")
               if os.path.exists(p)), URL)
print("reading", SOURCE)

df = pd.read_csv(SOURCE)
print(df.groupby('Battery').agg(cycles=('id_cycle','max'),
                                final_SoH=('SoH','min')).round(3))
df.head()

## 1. The physics

The **Wang throughput power law** (Wang et al., 2011):

$$\mathrm{SoH}(n) = 1 - c\,n^{z}$$

Two parameters, and the exponent is a *reading about the cell*, not just a knob:

- $z \approx 0.5$ — diffusion-limited SEI growth. Fade slows as the passivation layer thickens.
- $z \approx 1$ — linear wear. Something degrades at a constant rate.
- $z > 1$ — accelerating fade. The **knee**.

Fit it in log space, where the power law is linear and the fit is well posed:

$$\log(1 - \mathrm{SoH}) = \log c + z \log n$$

This matters more than it sounds. Fitting the raw curve on a short window is badly conditioned — capacity drops ~11 % while cycle-to-cycle noise is ~0.7 %, so $c$ and $z$ trade off almost freely and least squares wanders to whatever bound you set. Done that way these cells gave $z = 2.0$, sitting on the bound: not a physical reading, an optimiser lost in a flat valley.

In [ ]:
def wang(n, c, z):
    """SoH(n) = 1 - c * n**z"""
    return 1.0 - c * np.power(np.asarray(n, float), z)

def fit_physics(n, soh):
    m = (soh < 1.0) & (n > 0)
    z0, logc0 = np.polyfit(np.log(n[m]), np.log(1.0 - soh[m]), 1)  # log space
    popt, _ = curve_fit(wang, n, soh, p0=[np.exp(logc0), z0],
                        bounds=([1e-8, 0.3], [1e-1, 1.5]), maxfev=20000)
    return float(popt[0]), float(popt[1])

## 2. The data-only model

A Gaussian process on cycle number, given every advantage: a sensible kernel, a generous length scale, five restarts. Its failure below is not incompetence.

In [ ]:
def fit_gp(n, soh):
    kernel = (ConstantKernel(1.0, (1e-3, 1e3))
              * RBF(length_scale=40.0, length_scale_bounds=(5.0, 500.0))
              + WhiteKernel(1e-4, (1e-8, 1e-1)))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                  n_restarts_optimizer=5, random_state=SEED)
    return gp.fit(n.reshape(-1, 1), soh)

## 3. The hybrid

**The network never sees SoH.** It sees the *residual* — what the physics got wrong. That residual is small, roughly zero-mean and bounded, which is exactly why the hybrid cannot run away: even a badly extrapolating network is extrapolating a correction of order 1 %, on top of a trend that is still physically monotone.

Deliberately small — two hidden layers of 16, strongly regularised. A bigger network fits the training residual better and forecasts worse. That is the whole lesson, and it is easy to verify: change it to `(256, 256)` and re-run.

In [ ]:
def features(n):
    n = np.asarray(n, float).reshape(-1, 1)
    return np.hstack([n / 100.0, np.sqrt(n) / 10.0])

def fit_hybrid(n, soh, c, z):
    residual = soh - wang(n, c, z)          # <- the network's only target
    net = MLPRegressor(hidden_layer_sizes=(16, 16), activation='tanh',
                       alpha=1e-2, learning_rate_init=5e-3,
                       max_iter=8000, random_state=SEED)
    return net.fit(features(n), residual)

## 4. The honest test

Fit on the first 40 %. Forecast the rest. Compare against baselines that must be beaten: **persistence** (nothing changes) and **linear drift**.

In [ ]:
def rmse(y, yhat): return float(np.sqrt(np.mean((y - yhat) ** 2)))

def eol_cycle(n, soh, thr=EOL):
    below = np.where(soh <= thr)[0]
    if len(below) == 0 or below[0] == 0: return None
    i = below[0]; n0, n1, s0, s1 = n[i-1], n[i], soh[i-1], soh[i]
    return float(n1) if s0 == s1 else float(n0 + (thr-s0)*(n1-n0)/(s1-s0))

rows, curves = [], {}
for cell, g in df.groupby('Battery'):
    n = g.id_cycle.to_numpy(float); soh = g.SoH.to_numpy(float)
    k = int(len(n) * TRAIN_FRACTION)
    ntr, str_, nte, ste = n[:k], soh[:k], n[k:], soh[k:]

    c, z = fit_physics(ntr, str_)
    gp = fit_gp(ntr, str_); net = fit_hybrid(ntr, str_, c, z)

    preds = {'physics': wang(nte, c, z),
             'gp': gp.predict(nte.reshape(-1, 1)),
             'hybrid': wang(nte, c, z) + net.predict(features(nte)),
             'persistence': np.full(len(nte), str_[-1]),
             'drift': np.polyval(np.polyfit(ntr, str_, 1), nte)}
    base = rmse(ste, preds['persistence'])
    curves[cell] = (n, soh, k, preds)
    for m, p in preds.items():
        rows.append({'battery': cell, 'model': m, 'z': z,
                     'rmse': rmse(ste, p), 'skill': rmse(ste, p)/base})

res = pd.DataFrame(rows)
print(res.groupby('model')[['rmse','skill']].mean()
         .sort_values('rmse').round(4).to_string())

### The figure that carries the argument

In [ ]:
cell = 'B0005'
n, soh, k, preds = curves[cell]
split = n[k-1]
COL = {'physics':'#1C4E73','gp':'#B5651D','hybrid':'#2E7D32'}
LBL = {'physics':'Physics only','gp':'Data only (GP)','hybrid':'Hybrid'}

fig, (ax, axz) = plt.subplots(1, 2, figsize=(15, 6),
                              gridspec_kw={'width_ratios':[1.7,1]})
for a, lo in ((ax, n.min()), (axz, split + (n.max()-split)*0.30)):
    sel = n >= lo
    a.scatter(n[sel], soh[sel], s=18, color='#1B2430', alpha=.55, lw=0,
              label='Measured SoH' if a is ax else None, zorder=3)
    for m in ('physics','gp','hybrid'):
        msel = n[k:] >= lo
        a.plot(n[k:][msel], preds[m][msel], lw=2.6, color=COL[m],
               label=LBL[m] if a is ax else None, zorder=4)
    a.axhline(EOL, color='#B00020', lw=1.3, ls='--')
    a.set_xlabel('Discharge cycle')
ax.axvspan(n.min(), split, color='#EEF2F6', zorder=0)
ax.axvline(split, color='#66707A', lw=1.2)
ax.set_ylabel('State of Health'); ax.legend(loc='lower left', fontsize=10.5)
ax.set_title('fitted on the shaded window, forecasting everything right of it',
             loc='left', fontsize=12.5)
axz.set_title('the last third, up close', loc='left', fontsize=12.5)
fig.suptitle(f'Cell {cell} — the GP reverts to its mean; the physics keeps falling',
             x=0.01, ha='left', fontsize=15)
fig.tight_layout(rect=(0,0,1,0.95)); plt.show()

### Every cell, including the awkward ones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=True)
for a, (cell, (n, soh, k, preds)) in zip(axes.ravel(), curves.items()):
    a.axvspan(n.min(), n[k-1], color='#EEF2F6', zorder=0)
    a.scatter(n, soh, s=11, color='#1B2430', alpha=.5, lw=0, zorder=3)
    for m in ('physics','gp','hybrid'):
        a.plot(n[k:], preds[m], lw=2.1, color=COL[m], zorder=4)
    a.axhline(EOL, color='#B00020', lw=1.1, ls='--')
    a.set_title(cell, loc='left'); a.set_ylim(0.5, 1.03)
fig.suptitle('Four cells, one protocol', x=0.01, ha='left', fontsize=15)
fig.tight_layout(); plt.show()

### Skill score — below 1.0 beats doing nothing

In [ ]:
order = ['gp','physics','drift','hybrid']
means = res[res.model.isin(order)].groupby('model').skill.mean().reindex(order)
fig, a = plt.subplots(figsize=(10, 4.5))
bars = a.barh(order, means.values,
              color=['#B5651D','#1C4E73','#C0C4CC','#2E7D32'], height=.6)
a.axvline(1.0, color='#B00020', lw=1.6, ls='--')
a.text(1.02, -0.55, 'persistence baseline', color='#B00020', fontsize=10.5)
for b, v in zip(bars, means.values):
    a.text(v+.03, b.get_y()+b.get_height()/2, f'{v:.2f}', va='center',
           fontweight='bold')
a.set_xlabel('skill  =  model RMSE / persistence RMSE')
a.set_title('A Gaussian process on cycle number loses to assuming nothing changes',
            loc='left', fontsize=13)
plt.show()

## 5. What actually happened

Mean over four cells, temporal split at 40 %:

| Model | RMSE | Skill vs persistence |
|---|---|---|
| **Hybrid** | **0.0398** | **0.36** |
| Baseline: linear drift | 0.0490 | 0.44 |
| Physics only | 0.0544 | 0.50 |
| Baseline: persistence | 0.1109 | 1.00 |
| Data only (GP) | 0.1791 | 1.62 |

Three things worth sitting with:

**The Gaussian process is worse than assuming nothing changes.** Skill 1.62. Not because it is badly implemented — it is properly specified and fitted with restarts. Because outside the range it has seen, a GP reverts to its prior mean. It has no concept of a battery, so it has no reason to keep going down.

**A straight line beats the physics on RMSE.** Linear drift, skill 0.44 against the physics model's 0.50. That is humbling and it is real. Over a bounded horizon, extrapolating a line is a genuinely strong baseline, and a project that only reported its wins would have quietly dropped this row.

**But RMSE is not the question.** An operator asks *when do I replace it?* On that metric the ranking inverts:

| Model | Mean error in predicted end-of-life cycle |
|---|---|
| **Physics only** | **13.0 cycles** |
| Hybrid | 21.9 cycles |
| Baseline: linear drift | 25.7 cycles |

The straight line has the second-best RMSE and the worst answer to the actual question, because it crosses the 80 % line at the wrong angle. **Choose the metric that matches the decision, or you will optimise the wrong thing very precisely.**

**Try this:** change the network to `hidden_layer_sizes=(256, 256)` and re-run. The training residual fits beautifully. The forecast gets worse.

## 6. Where this goes next

This notebook is the *tutorial*. The same ideas, engineered properly, are a set of composable tools:

| Tool | What it does |
|---|---|
| [`otwin-systems`](https://github.com/otwin-core/otwin-systems) | physical model structures, each validated against a closed-form answer |
| [`otwin-eval`](https://github.com/otwin-core/otwin-eval) | the temporal split and mandatory baselines used here |
| [`otwin-uq`](https://github.com/otwin-core/otwin-uq) | calibrated uncertainty — this notebook has none, which is its biggest gap |
| [`otwin-phs`](https://github.com/otwin-core/otwin-phs) | port-Hamiltonian systems, for assets where the physics is an energy balance |

**What this notebook does not do, and should:** produce an interval. Every forecast above is a single line, and a single line is not a forecast — it is a guess with good posture. `otwin-uq` measures whether a 90 % band actually contains the truth 90 % of the time.

---

### The one thing to take away

The physics is not there for interpretability. It is there because it is the only part of the model that still knows what it is doing outside the data it was fitted on.

---

Full ecosystem: **[github.com/otwin-core](https://github.com/otwin-core)** · Apache 2.0